# S6E5 — Colab runner

Generic notebook for running any `notebooks/0X_*.py` training script on Colab Pro with GPU.

**Workflow** (run cells top to bottom):
1. Mount Google Drive (Drive folder `s6e5/` is the persistent artifact store)
2. Clone GitHub repo (latest code from `main`)
3. **Version meta** — verify which script will run (edit `SCRIPT` here)
4. Install dependencies (just-in-time, only what current version needs)
5. Sync data Drive → /content
6. Run the target script
7. Sync artifacts back to Drive
8. (optional) Download submission CSV to local machine

**Drive folder**: https://drive.google.com/drive/folders/1N6PFShEtMj2KSYWxaQQz-6Kro1CTLilh

**Verified path** (Drive MCP, 2026-05-12): `/content/drive/MyDrive/Colab Notebooks/kaggle/s6e5`

**Expected structure**:
```
MyDrive/Colab Notebooks/kaggle/s6e5/
├── data/
│   ├── raw/{train.csv, test.csv, sample_submission.csv}   ← MUST be real .csv, NOT Sheets
│   └── external/f1_strategy_dataset_v4.csv
├── probs/                                                  (auto-created)
├── submissions/                                            (auto-created)
└── experiments.jsonl                                       (synced from Colab runs)
```

⚠ **If Drive auto-converted the CSVs to Google Sheets** (files without `.csv` extension / green Sheets icon), the script will fail to read them. Fix: <https://drive.google.com/drive/settings> → uncheck **"Convert uploaded files to Google Docs editor format"** → delete + re-upload.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verified via Drive MCP 2026-05-12: folder lives at MyDrive/Colab Notebooks/kaggle/s6e5
# Note the space in 'Colab Notebooks' — DO NOT remove it; this is Google's auto-created folder name.
DRIVE_S6E5 = '/content/drive/MyDrive/Colab Notebooks/kaggle/s6e5'

!ls -la "$DRIVE_S6E5" || echo 'PATH NOT FOUND — adjust DRIVE_S6E5 above.'

# Sanity: data/raw/*.csv must be REAL csv files, not Google Sheets.
# If you see no '.csv' extension, the upload was converted to Sheets:
# fix at https://drive.google.com/drive/settings → uncheck 'Convert uploads' → re-upload.

## 2. Clone repo + check out latest

In [ ]:
%cd /content
!rm -rf playground-s6e5
!git clone -q https://github.com/SirGrigor/playground-s6e5.git
%cd playground-s6e5
!git log -1 --oneline

## 3. Version meta — verify which script will run

Edit `SCRIPT` below to pick which model version to train. Re-run this cell to see the verification block before consuming Colab compute.

In [ ]:
# === VERSION META — verify what we're about to run ===
SCRIPT = 'notebooks/04_v2_lgb_te.py'   # ← EDIT for v3, v4, etc.
EXTRA_ARGS = ''                         # e.g. '--gpu' or '--subsample 100000'

print("=" * 70)
print(f"ABOUT TO RUN:  {SCRIPT}")
print(f"EXTRA ARGS:    {EXTRA_ARGS or '(none)'}")
print("=" * 70)
print()
print("--- Latest commit on this branch ---")
!git log -1 --oneline
!git log -1 --format='committed %cd' --date=relative
print()
print(f"--- {SCRIPT} header (first 30 lines — hypothesis & predicted Δ) ---")
!head -30 {SCRIPT}
print()
print("=" * 70)
print("If this isn't the version you intended → edit SCRIPT above, re-run THIS cell.")
print("Otherwise → continue with the cells below.")
print("=" * 70)

## 4. Install dependencies

Just-in-time per version. v1 (LGB) and v2 (LGB+TE) only need `pyarrow` — Colab pre-installs the rest. v3/v5 will install catboost / pytabkit when they're needed.

In [ ]:
# Default to uv (per feedback_uv-over-pip). ONE pip call bootstraps uv,
# everything else uses uv pip.
#
# INSTALL ONLY WHAT CURRENT VERSION NEEDS — installing pytabkit/catboost
# eagerly drags in pytorch-lightning + numpy + scikit-learn upgrades that
# break Colab's pre-loaded sklearn C extensions.
#
# v1 (LGB):       only pyarrow         ← v1/v2 are here
# v2 (LGB+TE):    only pyarrow
# v3 (CatBoost):  add 'catboost' below
# v4 (HistGB):    (nothing extra — sklearn ships HistGB)
# v5 (RealMLP):   add 'pytabkit' below (accept dep churn at that point)

!pip install -q uv
!uv pip install -q --system pyarrow

# Verify only what we need
import lightgbm, sklearn, numpy, pandas, pyarrow
print(f'numpy {numpy.__version__}  pandas {pandas.__version__}  sklearn {sklearn.__version__}')
print(f'lgb {lightgbm.__version__}  pyarrow {pyarrow.__version__}')

# CV-safe TargetEncoder is in sklearn >= 1.3
assert tuple(int(x) for x in sklearn.__version__.split('.')[:2]) >= (1, 3), \
    f"sklearn {sklearn.__version__} is too old — need >= 1.3 for CV-safe TargetEncoder"
print("\n✓ sklearn version supports CV-safe TargetEncoder")

# GPU check (LGB runs fine on CPU at 350K rows but T4 is faster)
import torch
print(f'\ntorch {torch.__version__}  CUDA={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

## 5. Sync data Drive → /content

Drive IO is slow — copy once per Colab session.

In [ ]:
import os, shutil

os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/external', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)
os.makedirs('probs', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

# raw competition data
for fn in ('train.csv', 'test.csv', 'sample_submission.csv'):
    src = f'{DRIVE_S6E5}/data/raw/{fn}'
    dst = f'data/raw/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')
    else:
        print(f'  already present: {fn}')

# external dataset
for fn in ('f1_strategy_dataset_v4.csv',):
    src = f'{DRIVE_S6E5}/data/external/{fn}'
    dst = f'data/external/{fn}'
    if not os.path.exists(dst):
        shutil.copyfile(src, dst)
        print(f'  copied {fn}')
    else:
        print(f'  already present: {fn}')

# sacred holdout split is in git already (data/splits/holdout_v1.parquet) — no copy needed
print('\n--- Verifying files ---')
!ls -la data/raw data/external data/splits

## 6. Run the target script

Uses `SCRIPT` and `EXTRA_ARGS` from the **Version meta** cell above. Don't edit anything here — edit Version meta then come back.

In [ ]:
print(f"RUNNING: {SCRIPT}  {EXTRA_ARGS}")
print(f"cwd: {os.getcwd()}")
!python {SCRIPT} {EXTRA_ARGS}

## 7. Sync artifacts /content → Drive

Persist outputs back so they survive Colab session shutdown.

In [ ]:
import os, shutil

if not os.path.isdir('probs'):
    print('ABORT: probs/ does not exist in cwd. The run script in cell 6 did not produce artifacts.')
    print('Check the run cell output for errors before re-running this cell.')
else:
    # Sync each probs/version_dir to Drive
    for d in sorted(os.listdir('probs')):
        src = f'probs/{d}'
        dst = f'{DRIVE_S6E5}/probs/{d}'
        os.makedirs(dst, exist_ok=True)
        for fn in os.listdir(src):
            shutil.copyfile(f'{src}/{fn}', f'{dst}/{fn}')
        print(f'  synced probs/{d}/ → Drive')

    # submissions
    if os.path.isdir('submissions'):
        os.makedirs(f'{DRIVE_S6E5}/submissions', exist_ok=True)
        for fn in sorted(os.listdir('submissions')):
            shutil.copyfile(f'submissions/{fn}', f'{DRIVE_S6E5}/submissions/{fn}')
            print(f'  synced submissions/{fn} → Drive')

    # experiments.jsonl backup to Drive (git-tracked too, but Drive is failsafe)
    if os.path.exists('experiments.jsonl'):
        shutil.copyfile('experiments.jsonl', f'{DRIVE_S6E5}/experiments.jsonl')
        print('  synced experiments.jsonl → Drive')

    print('\n✓ sync complete.')

## 8. Download submission CSV to local machine

Use this for the FINAL submission of each version. The submission then goes via `kaggle competitions submit` from your local machine.

In [ ]:
# Build path from SCRIPT name. e.g. 'notebooks/04_v2_lgb_te.py' → 'submissions/v2_lgb_te.csv'
from pathlib import Path
stem = Path(SCRIPT).stem.split('_', 1)[1]   # '04_v2_lgb_te' → 'v2_lgb_te'
sub_path = f'submissions/{stem}.csv'

from google.colab import files
files.download(sub_path)